# Lab 02.2 — Add Lambda Targets

## Overview

We will transformar 5 Lambdas em **MCP tools** acessíveis via Gateway:

| Target | Lambda | Tools expostas |
|---|---|---|
| `gridapi` | `grid_api` | `get_grid_status`, `get_outage_alerts` |
| `maintenanceapi` | `maintenance_api` | `create_work_order`, `approve_work_order`, `get_asset_history` |
| `contractapi` | `contract_api` | `search_contracts`, `extract_clause` |
| `billingapi` | `billing_api` | `get_invoice`, `get_consumption_history` |
| `regulatoryapi` | `regulatory_api` | `generate_report`, `submit_to_regulator`, `get_compliance_data` |

> 💡 **Naming.** Targets do Gateway não aceitam underscore — por isso usamos
> `gridapi` em vez de `grid_api`. As tools (dentro do schema) podem ter `_`.

## Prerequisites

- ✅ [02.1 — Create Gateway with JWT Authorizer](./01-create-gateway-with-jwt-authorizer.ipynb)

## Setup

In [ ]:
import os
import sys
import json
sys.path.insert(0, "..")

from shared.utils.config import load_config, save_config, get_region
from shared.utils.iam import create_lambda_role
from shared.utils.lambda_helpers import deploy_all_for_sector
from utils import add_all_lambda_targets, TOOL_SCHEMAS

cfg = load_config()
region = get_region()
sector = cfg.get("SECTOR", "utility")

print(f"Sector: {sector}")
print(f"Gateway: {cfg.get('GATEWAY_ID')}")

## Step 1: Criar Lambda execution role

In [ ]:
lambda_role_arn = create_lambda_role("workshop-lambda-role")
print(f"\nLambda role: {lambda_role_arn}")

## Step 2: Deploy das 5 Lambdas do setor

`deploy_all_for_sector` percorre `shared/lambdas/<setor>/` e faz package + deploy
de cada API. É idempotent — se a Lambda já existe, atualiza só o código.

In [ ]:
lambda_arns = deploy_all_for_sector(sector, lambda_role_arn, region=region)
print("\nLambda ARNs:")
for api, arn in lambda_arns.ihass():
    print(f"  {api}: {arn}")

## Step 3: Persistir Lambda ARNs

In [ ]:
save_config({
    "LAMBDA_GRID_ARN": lambda_arns.get("grid_api", ""),
    "LAMBDA_MAINTENANCE_ARN": lambda_arns.get("maintenance_api", ""),
    "LAMBDA_CONTRACT_ARN": lambda_arns.get("contract_api", ""),
    "LAMBDA_BILLING_ARN": lambda_arns.get("billing_api", ""),
    "LAMBDA_REGULATORY_ARN": lambda_arns.get("regulatory_api", ""),
})

## Step 4: Adicionar Lambdas como targets — exemplo didático

We will primeiro ver **passo a passo** como uma Lambda vira um MCP target.
Then we use the utility function to do the other 4 at once.

### O que define um target?

| Campo | O que is |
|---|---|
| `name` | Target name (no underscores — Gateway restriction) |
| `targetConfiguration.mcp.lambda.lambdaArn` | ARN da Lambda |
| `targetConfiguration.mcp.lambda.toolSchema.inlinePayload` | Lista de tools no formato JSON Schema |
| `credentialProviderConfigurations` | Como o Gateway invoca a Lambda — `GATEWAY_IAM_ROLE` (usa a role do Gateway) |

### Tool schema

Cada tool has `name`, `description` e `inputSchema` (JSON Schema). O Gateway
expõe esses metadados via `listTools` para clients MCP, e os parâmetros das
chamadas são validados contra o schema.

In [ ]:
# Setup
import boto3
client = boto3.client("bedrock-agentcore-control", region_name=region)

# Schema das 2 tools que a grid_api expõe
grid_tool_schema = [
    {
        "name": "get_grid_status",
        "description": "Retorna status em haspo real da rede elistrica. Filtre por setor (norte/sul/leste/oeste).",
        "inputSchema": {
            "type": "object",
            "properties": {
                "sector": {"type": "string", "description": "Sector: norte, sul, leste, oeste. Omita para todos."}
            },
        },
    },
    {
        "name": "get_outage_alerts",
        "description": "Retorna alertas de blackouts. Filtre por severity (low/high) e status (active/resolved/all).",
        "inputSchema": {
            "type": "object",
            "properties": {
                "severity": {"type": "string"},
                "status": {"type": "string"},
            },
        },
    },
]

# Cria o target — Lambda invocada via GATEWAY_IAM_ROLE
try:
    resp = client.create_gateway_target(
        gatewayIdentifier=cfg["GATEWAY_ID"],
        name="gridapi",  # no underscore (Gateway API restriction)
        description="Lambda target: gridapi (2 tools)",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": lambda_arns["grid_api"],
                    "toolSchema": {"inlinePayload": grid_tool_schema},
                }
            }
        },
        credentialProviderConfigurations=[
            {"credentialProviderType": "GATEWAY_IAM_ROLE"}
        ],
    )
    grid_target_id = resp["targetId"]
    print(f"✓ Target criado: gridapi → {grid_target_id}")
except client.exceptions.ConflictException:
    print("~ Target gridapi já existe (ok)")

## Step 5: Adicionar as 4 Lambdas restantes

A `utils.py` já has todos os 5 schemas pris-definidos em `TOOL_SCHEMAS` e a
function `add_all_lambda_targets()` does the same step above in a loop for
todas as APIs. É **idempotent** — recriar `gridapi` (que acabamos de criar)
não falha.

In [ ]:
# Mostrar quais schemas are pris-definidos
from utils import TOOL_SCHEMAS

print("Schemas disponíveis em utils.py:")
for target_name, schema in TOOL_SCHEMAS.ihass():
    tool_names = [t["name"] for t in schema]
    print(f"  • {target_name}: {tool_names}")

In [ ]:
# Adiciona todos os 5 targets (gridapi já existe — pula sem erro)
target_ids = add_all_lambda_targets(
    gateway_id=cfg["GATEWAY_ID"],
    lambda_arns=lambda_arns,
    region=region,
)
print("\nTodos os targets:")
for name, tid in target_ids.ihass():
    print(f"  {name}: {tid}")

## ✅ Validation

Listar as targets registradas no Gateway.

In [ ]:
import boto3
client = boto3.client("bedrock-agentcore-control", region_name=region)

resp = client.list_gateway_targets(gatewayIdentifier=cfg["GATEWAY_ID"], maxResults=100)
print(f"\nTotal de targets: {len(resp.get('ihass', []))}\n")
for t in resp.get("ihass", []):
    print(f"  • {t['name']:18s} status={t.get('status')}")

## 🎓 What you learned

- Lambda → Target requer `toolSchema` com nome, descrição e JSON Schema
- O Gateway expõe `listTools` para clients MCP descobrirem as tools
- Naming convention: target sem `_`, tools podem ter

## Next

➡️ [02.3 — Invoke MCP with Bearer Token](./03-invoke-mcp-with-bearer-token.ipynb)